In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.nn import functional as F

# **Parameter Management**

In [2]:
net = nn.Sequential(nn.LazyLinear(8),
                    nn.ReLU(),
                    nn.LazyLinear(1))

X = torch.rand(size=(2, 4))
net(X).shape

torch.Size([2, 1])

In [3]:
# Parameter Access
net[2].state_dict()

OrderedDict([('weight',
              tensor([[ 0.0893, -0.1919, -0.2232, -0.1340, -0.1662, -0.0947,  0.2207,  0.1032]])),
             ('bias', tensor([0.1669]))])

In [4]:
# Targeted Parameters
type(net[2].bias), net[2].bias.data

(torch.nn.parameter.Parameter, tensor([0.1669]))

In [5]:
net[2].weight.grad == None

True

In [6]:
# All Parameters at Once
[(name, param.shape) for name, param in net.named_parameters()]

[('0.weight', torch.Size([8, 4])),
 ('0.bias', torch.Size([8])),
 ('2.weight', torch.Size([1, 8])),
 ('2.bias', torch.Size([1]))]

In [7]:
# Tied parameters

shared = nn.LazyLinear(8)
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(),
                    shared, nn.ReLU(),
                    shared, nn.ReLU(),
                    nn.LazyLinear(1))

net(X)

print(net[2].weight.data[0] == net[4].weight.data[0])
net[2].weight.data[0, 0] = 100
print(net[2].weight.data[0] == net[4].weight.data[0])

tensor([True, True, True, True, True, True, True, True])
tensor([True, True, True, True, True, True, True, True])


# **Parameter Initialization**

In [8]:
net = nn.Sequential(nn.LazyLinear(8), nn.ReLU(), nn.LazyLinear(1))
X = torch.rand(size=(2, 4))
net(X).shape

torch.Size([2, 1])

In [9]:
def init_normal(module):
    if type(module) == nn.Linear:
        nn.init.normal_(module.weight, mean=0, std=0.01)
        nn.init.zeros_(module.bias)

net.apply(init_normal)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([-0.0015, -0.0111, -0.0116, -0.0053]), tensor(0.))

In [10]:
def init_constant(module):
    if type(module) == nn.Linear:
        nn.init.constant_(module.weight, 1)
        nn.init.zeros_(module.bias)

net.apply(init_constant)
net[0].weight.data[0], net[0].bias.data[0]

(tensor([1., 1., 1., 1.]), tensor(0.))

In [11]:
def init_xavier(module):
    if type(module) == nn.Linear:
        nn.init.xavier_uniform_(module.weight)

def init_constant_42(module):
    if type(module) == nn.Linear:
        nn.init.constant_(module.weight, 42)

net[0].apply(init_xavier)
net[2].apply(init_constant_42)
print(net[0].weight.data[0])
print(net[2].weight.data[0])

tensor([-0.4373, -0.4492, -0.2887,  0.6529])
tensor([42., 42., 42., 42., 42., 42., 42., 42.])


In [12]:
# Custom Initialization
def my_init(module):
    if type(module) == nn.Linear:
        print('Init', *[(name, param.shape) for name, param in module.named_parameters()][0])
        nn.init.uniform_(module.weight, -10, 10)
        module.weight.data *= module.weight.data.abs() >= 5

net.apply(my_init)
net[0].weight[:2]

Init weight torch.Size([8, 4])
Init weight torch.Size([1, 8])


tensor([[-7.5184, -5.0527, -0.0000,  0.0000],
        [ 8.1999, -7.2139,  0.0000,  8.5584]], grad_fn=<SliceBackward0>)

In [13]:
net[0].weight.data[:] += 1
net[0].weight.data[0, 0] = 42
net[0].weight.data[0]

tensor([42.0000, -4.0527,  1.0000,  1.0000])

# **Lazy Initialization**

In [14]:
net = nn.Sequential(nn.LazyLinear(256), nn.ReLU(), nn.LazyLinear(10))

In [15]:
net[0].weight

<UninitializedParameter>

In [16]:
X = torch.rand(2, 20)
net(X)

net[0].weight.shape

torch.Size([256, 20])

# **Custom Layers**

In [17]:
# Layers without Parameters
class CenteredLayer(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, X):
        return X - X.mean()

In [18]:
layer = CenteredLayer()
layer(torch.tensor([1.0, 2, 3, 4, 5]))

tensor([-2., -1.,  0.,  1.,  2.])

In [19]:
net = nn.Sequential(nn.LazyLinear(128), CenteredLayer())

In [20]:
Y = net(torch.rand(4, 8))
Y.mean()

tensor(2.7940e-09, grad_fn=<MeanBackward0>)

In [21]:
# Layers with Parameters
class MyLinear(nn.Module):
    def __init__(self, in_units, units):
        super().__init__()
        self.weight = nn.Parameter(torch.randn(in_units, units))
        self.bias = nn.Parameter(torch.randn(units,))

    
    def forward(self, X):
        linear = torch.matmul(X, self.weight.data) + self.bias.data
        return F.relu(linear)

In [22]:
linear = MyLinear(5, 3)
linear.weight

Parameter containing:
tensor([[-0.2355,  0.3518,  0.1848],
        [ 0.9888,  0.1245, -0.3226],
        [ 1.0003,  0.2753, -0.0356],
        [ 0.1239, -0.3705,  0.7942],
        [-0.6903,  0.5375,  0.0845]], requires_grad=True)

In [23]:
linear(torch.rand(2, 5))

tensor([[1.0736, 0.5575, 0.7045],
        [0.7613, 0.0000, 1.2403]])

In [24]:
net = nn.Sequential(MyLinear(64, 8), MyLinear(8, 1))
net(torch.rand(2, 64))

tensor([[0.],
        [0.]])

# **File I/O**

In [25]:
# Loading and Saving Tensors
x = torch.arange(4)
torch.save(x, 'x-file')

In [26]:
x2 = torch.load('x-file')
x2

tensor([0, 1, 2, 3])

In [27]:
y = torch.zeros(4)
torch.save([x, y], 'x-files')
x2, y2 = torch.load('x-files')
(x2, y2)

(tensor([0, 1, 2, 3]), tensor([0., 0., 0., 0.]))

In [28]:
mydict = {'x': x, 'y': y}
torch.save(mydict, 'mydict')
mydict2 = torch.load('mydict')
mydict2

{'x': tensor([0, 1, 2, 3]), 'y': tensor([0., 0., 0., 0.])}

In [30]:
# Loading and Saving Model Parameters
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.LazyLinear(256)
        self.output = nn.LazyLinear(10)

    def forward(self, X):
        return self.output(F.relu(self.hidden(X)))
    
net = MLP()
X = torch.randn(size=(2, 20))
Y = net(X)

In [31]:
torch.save(net.state_dict(), 'mlp.params')

In [32]:
clone = MLP()
clone.load_state_dict(torch.load('mlp.params'))
clone.eval()

MLP(
  (hidden): LazyLinear(in_features=0, out_features=256, bias=True)
  (output): LazyLinear(in_features=0, out_features=10, bias=True)
)

In [33]:
Y_clone = clone(X)
Y_clone == Y

tensor([[True, True, True, True, True, True, True, True, True, True],
        [True, True, True, True, True, True, True, True, True, True]])

# **GPUs**

In [ ]:
# Computing devices
def cpu():
    """Get the CPU device."""
    return torch.device('cpu')

def gpu(i=0):
    """Get a GPU device."""
    return torch.device(f'cuda:{i}')

cpu(), gpu(), gpu(1)

(device(type='cpu'),
 device(type='cuda', index=0),
 device(type='cuda', index=1))

In [35]:
def num_gpus():
    """Get the number of available GPUs."""
    return torch.cuda.device_count()

num_gpus()

1

In [37]:
def try_gpu(i=0):
    """Return gpu(i) if exists, otherwise return cpu()."""
    if num_gpus() >= i + 1:
        return gpu(i)
    return cpu()

def try_all_gpus():
    """Return all available GPUs, or [cpu(),] if no GPU exists."""
    return [gpu(i) for i in range(num_gpus())]

try_gpu(), try_gpu(10), try_all_gpus()

(device(type='cuda', index=0),
 device(type='cpu'),
 [device(type='cuda', index=0)])

In [38]:
# Tensors and GPUs
x = torch.tensor([1, 2, 3])
x.device

device(type='cpu')

In [39]:
# Storage on the GPU
X = torch.ones(2, 3, device=try_gpu())
X

tensor([[1., 1., 1.],
        [1., 1., 1.]], device='cuda:0')

In [40]:
Y = torch.rand(2, 3, device=try_gpu(1))
Y

tensor([[0.0377, 0.2971, 0.0352],
        [0.4464, 0.7348, 0.2827]])

In [41]:
net = nn.Sequential(nn.LazyLinear(1))
net = net.to(device=try_gpu())

In [42]:
net(X)

tensor([[0.7205],
        [0.7205]], device='cuda:0', grad_fn=<AddmmBackward0>)